# Quickstart

This notebook shows the shortest path from the included demo photometry table to a saved accretion-disk fit. The demo file is a small 2025 slice of the development table, so you can run the workflow immediately after installation. Replace `data_path` with your own file when you are ready.


## 1. Imports

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "accretion").exists()),
    Path.cwd(),
)
SRC_PATH = PROJECT_ROOT / "src"
for path in (SRC_PATH,):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

# Optional: For better graphics in Jupyter notebooks
%matplotlib widget


In [ ]:
from pathlib import Path

import pandas as pd

from accretion import (
    basic_fitting,
    load_database_new,
    load_irtf_lxd_spectra,
    load_irtf_sxd_spectra,
    rede_fitting,
    result_saver,
    spectral_fitting,
)
from accretion.final_panels import plot_filter_evolution_panel, plot_final_residuals
from accretion.final_panel_colour_grid import plot_final_panel_colour_grid

## 2. Load Data

The demo file uses data from the young eruptive star V34NOR. For your own data, use `pd.read_csv` instead.


In [ ]:
data_path = Path("../data/demo_data.txt")

if not data_path.exists():
    raise FileNotFoundError(f"Could not find demo data at {data_path}")

df = load_database_new(data_path)

# For your own CSV, use:
# data_path = Path("../data/raw/my_photometry.csv")
# df = pd.read_csv(data_path)

df.head()


## 3. Run A Basic JHK Fit

The demo data contains enough `J`, `H`, and `K` measurements to run the basic mode on a small number of observing days. However you can run your own fits on any other photometric data.


In [ ]:
basic_results = basic_fitting(
    df,
    required_filters=("J", "H", "K"),
    fit_filters=("J", "H", "K"),
    lambda_reg=0,
    debug=False,
)

result_saver(basic_results, filepath="../results/generated/datas/basic_JHK")

## 4. Optional: Run Red-Excess Mode

This mode adds a blackbody red-excess component to the model fitted to the data. Fitting the BB to the 'red' region and accretion disk model to the 'blue'.

In [ ]:
rede_results = rede_fitting(
    df,
    required_filters=("J", "H", "K", "L"),
    fit_filters=("J", "H", "K", "L"),
    filename="red_excess",
    lambda_reg=1,
)
result_saver(rede_results, filepath="../results/generated/datas/red_excess")


## 5. Optional: Run Spectral Mode

Use these cells if your data is spectra rather than filter photometry.

In [ ]:
# sxd = load_irtf_sxd_spectra("../data/raw/IRTF")
# lxd = load_irtf_lxd_spectra("../data/raw/IRTF")
# spectra = pd.concat([sxd, lxd], ignore_index=True)
# spectral_results = spectral_fitting(
#     spectra,
#     lambda_reg=1,
#     rede=True,
#     save_dir="../results/generated/pictures/spectral_red_excess",
# )
# result_saver(spectral_results, filepath="../results/generated/datas/spectral_red_excess")

## 6. Visualisations

These examples reuse the basic JHK fit above. The plotting options below are the useful switches to tweak when you try your own data. Figures are saved as PDFs in `results/generated/pictures/`.


In [ ]:
# Shared plotting options.
VISUAL_FILTERS = ("J", "H", "K")
RESIDUAL_FILTER_GROUPS = ("I", "J", "H", "K", "L+W1")
DATE_RANGE = (None, None)  # Example: (2459000, 2461200)

SHOW_MODEL_FIT = True
DIFFERENTIATE_FILTER_LAMBDAS = False

### 6.1 Flexible Filter Evolution

This is the standalone version of the first final-panel plot: observed photometry with optional model overlay, filter/lambda splitting, and date limits.


In [ ]:
filter_fig, filter_ax = plot_filter_evolution_panel(
    df,
    basic_results,
    selected_filters=VISUAL_FILTERS,
    plot_fits=SHOW_MODEL_FIT,
    filter_diff=DIFFERENTIATE_FILTER_LAMBDAS,
    date_range=DATE_RANGE,
    save_path="quickstart_filter_evolution",
    show=True,
)


### 6.2 Combined Residuals

This uses the final-panel residual machinery: all selected residual groups in one figure, with magnitude residuals and flux residuals sharing colours.


In [ ]:
residual_fig, residual_axes, residual_tables = plot_final_residuals(
    df,
    basic_results,
    fit_filters=VISUAL_FILTERS,
    residual_filter_groups=RESIDUAL_FILTER_GROUPS,
    show_magnitude=True,
    show_flux=True,
    date_range=DATE_RANGE,
    save_path="quickstart_residuals",
    show=True,
)


### 6.3 Colour And Colour-Magnitude Plots

The colour-grid helper can compare observed colours with model colours. Here we show `I` vs `I-J` and `J-H` vs `H-K`.


In [ ]:
colour_fig, colour_matches, colour_models = plot_final_panel_colour_grid(
    df,
    basic_results,
    tolerance=2.0,
    show_fit=False,
    plot=("I,I-J", "J-H,H-K"),
    save_path="quickstart_colour_examples",
    show=True,
)

{name: len(matches) for name, matches in colour_matches.items()}
